In [66]:
from ast import literal_eval

import polars as pl

In [10]:
negocios = pl.read_csv("data/negocios.csv")
print(negocios.shape)
print(negocios.schema)
negocios.show(5)

(30069, 14)
Schema({'business_id': String, 'name': String, 'address': String, 'city': String, 'state': String, 'postal_code': String, 'latitude': Float64, 'longitude': Float64, 'stars': Float64, 'review_count': Int64, 'is_open': Int64, 'attributes': String, 'categories': String, 'hours': String})


business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
str,str,str,str,str,str,f64,f64,f64,i64,i64,str,str,str
"""GDEEPQdYs2utMN-R4znZSA""","""Metro Self Storage - Largo""","""10501 S Belcher Rd""","""Largo""","""FL""","""33777""",27.868519,-82.743849,4.5,7,0,"""{'BusinessAcceptsCreditCards':…","""Packing Supplies, Shopping, Lo…","""{'Monday': '0:0-0:0', 'Tuesday…"
"""pbAq2NRG_2jCBI6fgRalvQ""","""Madewell""","""3301 Veterans Blvd, Ste 84""","""Metairie""","""LA""","""70002""",30.005894,-90.15745,2.0,6,1,"""{'BusinessAcceptsCreditCards':…","""Shopping, Women's Clothing, Fa…","""{'Monday': '10:0-21:0', 'Tuesd…"
"""h4HP3Vc0dQq7SfSLal9qQw""","""Dollylocks""","""511 9th St N""","""Saint Petersburg""","""FL""","""33701""",27.777785,-82.646673,4.5,7,0,"""{'RestaurantsPriceRange2': '4'…","""Hair Stylists, Beauty & Spas, …",null
"""PndbFVbHE4730HDlghxv6g""","""Jim Browne Chrysler Jeep Dodge…","""10909 N Florida Ave""","""Tampa""","""FL""","""33612""",28.048093,-82.458757,2.5,76,1,"""{'BusinessAcceptsCreditCards':…","""Automotive, Auto Parts & Suppl…","""{'Monday': '7:30-20:0', 'Tuesd…"
"""IayDnngl0NooAbcoo62j-w""","""Emg Salons""","""324 S Falkenburg Rd""","""Tampa""","""FL""","""33619""",27.948815,-82.334619,4.5,7,1,"""{'WiFi': ""u'free'"", 'BikeParki…","""Hair Stylists, Hair Salons, Be…","""{'Tuesday': '9:0-20:0', 'Wedne…"


In [11]:
negocios.select([
    pl.all().null_count()
])

business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,1001,0,0,12,0,0,0,0,0,2769,17,4688


In [12]:
negocios.select([
    pl.col("city").n_unique(),
    pl.col("state").n_unique(),
    pl.col("categories").n_unique()
])

city,state,categories
u32,u32,u32
776,17,19691


In [14]:
negocios.select([
    pl.col("stars").mean().alias("avg_stars"),
    pl.col("stars").min().alias("min_stars"),
    pl.col("stars").max().alias("max_stars"),
    pl.col("review_count").mean().alias("avg_review_count"),
    pl.col("review_count").median().alias("median_review_count"),
    pl.col("review_count").max().alias("max_review_count")
])

avg_stars,min_stars,max_stars,avg_review_count,median_review_count,max_review_count
f64,f64,f64,f64,f64,i64
3.589511,1.0,5.0,44.372576,15.0,7568


In [29]:
negocios.select([
    pl.col("stars").value_counts().sort()  
]).unnest("stars")

stars,count
f64,u32
1.0,407
1.5,991
2.0,1998
2.5,2842
3.0,3684
3.5,5282
4.0,6221
4.5,5409
5.0,3235


In [43]:
negocios.select([
    pl.col("review_count").quantile([0.25, 0.5, 0.75, 0.9, 0.99])
])

review_count
list[f64]
"[8.0, 15.0, … 461.0]"


In [49]:
negocios.group_by("is_open").agg([
    pl.len().alias("num_businesses"),
    pl.col("stars").mean().alias("avg_stars"),
    pl.col("review_count").mean().alias("avg_reviews")
])

is_open,num_businesses,avg_stars,avg_reviews
i64,u32,f64,f64
1,23932,3.612841,46.233829
0,6137,3.498533,37.114388


In [51]:
negocios.group_by("state").agg([
    pl.len().alias("num_businesses")
]).sort("num_businesses", descending=True)

state,num_businesses
str,u32
"""PA""",6852
"""FL""",5226
"""TN""",2363
"""IN""",2231
"""MO""",2152
…,…
"""DE""",436
"""IL""",376
"""CO""",1


In [52]:
negocios.group_by("state").agg([
    pl.len().alias("num_businesses")
]).sort("num_businesses", descending=True)

state,num_businesses
str,u32
"""PA""",6852
"""FL""",5226
"""TN""",2363
"""IN""",2231
"""MO""",2152
…,…
"""DE""",436
"""IL""",376
"""CO""",1


In [55]:
negocios = negocios.with_columns(
    pl.col("categories")
    .str.split(", ")
    .alias("categories_list")
)

negocios.explode("categories_list") \
    .group_by("categories_list") \
    .agg(pl.len().alias("count")) \
    .sort("count", descending=True) \
    .head(20)

categories_list,count
str,u32
"""Restaurants""",10457
"""Food""",5575
"""Shopping""",4868
"""Beauty & Spas""",2921
"""Home Services""",2782
…,…
"""Coffee & Tea""",1337
"""Fast Food""",1307
"""American (New)""",1214


In [57]:
negocios.explode("categories_list") \
    .group_by("categories_list") \
    .agg([
        pl.len().alias("count"),
        pl.col("stars").mean().alias("avg_stars")
    ]) \
    .filter(pl.col("count") > 50) \
    .sort("avg_stars", descending=True)

categories_list,count,avg_stars
str,u32,f64
"""Chiropractors""",152,4.546053
"""Pilates""",65,4.530769
"""Event Photography""",62,4.483871
"""Boot Camps""",64,4.46875
"""Acupuncture""",80,4.4625
…,…,…
"""Fast Food""",1307,2.610941
"""Post Offices""",61,2.598361
"""Apartments""",361,2.570637


In [63]:
negocios.select(pl.col("latitude"), pl.col("longitude")).describe()

statistic,latitude,longitude
str,f64,f64
"""count""",30069.0,30069.0
"""null_count""",0.0,0.0
"""mean""",36.699153,-89.347668
"""std""",5.90256,14.968181
"""min""",27.558259,-120.092141
"""25%""",32.179394,-90.35933
"""50%""",38.79036,-86.119031
"""75%""",39.955445,-75.393805
"""max""",53.647207,-74.658572


In [64]:
negocios.select([
    pl.corr("stars", "review_count")
])

stars
f64
0.063282


In [65]:
negocios.sort("review_count", descending=True).select([
    "name", "stars", "review_count", "city"
]).head(10)

name,stars,review_count,city
str,f64,i64,str
"""Acme Oyster House""",4.0,7568,"""New Orleans"""
"""Reading Terminal Market""",4.5,5721,"""Philadelphia"""
"""Commander's Palace""",4.5,4876,"""New Orleans"""
"""Cochon Butcher""",4.5,3837,"""New Orleans"""
"""El Vez""",4.0,3187,"""Philadelphia"""
"""Coop's Place""",3.5,3157,"""New Orleans"""
"""Café Amelie""",4.5,2756,"""New Orleans"""
"""Puckett's Grocery & Restaurant""",4.0,2746,"""Nashville"""
"""Boathouse at Hendry's Beach""",4.0,2536,"""Santa Barbara"""


In [76]:
def parse_attributes(attr_str):
    if attr_str is None:
        return None
    try:
        return literal_eval(attr_str)
    except Exception as e:
        print(f"Error parsing attributes: {e}. Input: {attr_str}")
        return None
    
negocios.with_columns(
    pl.col("attributes").map_elements(parse_attributes).alias("attributes_dict")
).select("attributes")

Error parsing attributes: invalid syntax (<unknown>, line 0). Input: 
Error parsing attributes: invalid syntax (<unknown>, line 0). Input: 


attributes
str
"""{'BusinessAcceptsCreditCards':…"
"""{'BusinessAcceptsCreditCards':…"
"""{'RestaurantsPriceRange2': '4'…"
"""{'BusinessAcceptsCreditCards':…"
"""{'WiFi': ""u'free'"", 'BikeParki…"
…
"""{'Ambience': ""{'touristy': Fal…"
"""{'ByAppointmentOnly': 'False',…"
"""{'BusinessAcceptsCreditCards':…"
